# British Pathé — Turin Scraper

British Pathé is a historical film archive containing thousands of newsreels and documentary clips dating from 1896 to 1984. For this project I scraped all videos tagged with **Turin** to build a dataset of archival footage of the city — industrial scenes, public events, labour history, and everyday life across the 20th century.

The scraper does four things:
1. Iterates through all paginated search results for the query "Turin"
2. Extracts metadata for each clip: title, video page URL, direct mp4 link, thumbnail, and summary
3. Downloads the thumbnail images
4. Downloads the video previews using authenticated browser cookies

I used Selenium because the site renders its search results dynamically via JavaScript — a plain requests-based scraper would receive an empty page.

**Note on re-running: This scraper was originally run in November 2025 and collected 205 results. Since then, British Pathé has added Cloudflare bot protection to its search pages, which blocks automated browsers before any content loads. Attempts to bypass this using `undetected-chromedriver` also failed due to a version mismatch between the patched driver and the installed Chrome version. The original dataset (205 clips, thumbnails, and video files) is preserved on OneDrive and referenced throughout this report. The code below represents the working pipeline used to collect that data.**

All outputs are saved in the same folder as this notebook:
- `britishpathe_turin_full.csv`
- `thumbnails_turin/`
- `videos_turin/`

## 1. Setup

All outputs are saved relative to the notebook's location using `os.path.dirname(os.path.abspath('__file__'))`. This means wherever the notebook lives, the CSV and folders will always be created next to it. The `HEADLESS` toggle switches between a visible and background browser.

In [ ]:
!pip install selenium webdriver-manager pandas requests -q

In [ ]:
import os, time, re
import pandas as pd
import requests
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager

# All outputs saved next to this notebook
HERE       = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\BRITISH PATHE"

HEADLESS   = False
BASE_URL   = "https://www.britishpathe.com/search/?searchQuery=turin"
OUTPUT_CSV = os.path.join(HERE, "britishpathe_turin_full.csv")
THUMBS_DIR = os.path.join(HERE, "thumbnails_turin")
VIDEOS_DIR = os.path.join(HERE, "videos_turin")

os.makedirs(THUMBS_DIR, exist_ok=True)
os.makedirs(VIDEOS_DIR, exist_ok=True)

print("Output CSV :", OUTPUT_CSV)
print("Thumbnails :", THUMBS_DIR)
print("Videos     :", VIDEOS_DIR)

## 2. Browser driver

I use `webdriver-manager` to automatically download the correct ChromeDriver version for whatever Chrome is installed — this avoids the version mismatch errors that occur when Chrome auto-updates. A custom user-agent and disabled automation flag reduce the chance of the site blocking the browser.

In [ ]:
def make_driver(headless: bool = True):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--window-size=1600,1200")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    return driver

## 3. Scraping the search results

The search results are paginated — each page is accessed by appending `&page=N` to the URL. For each video card I extract the title, video page URL, direct mp4 link, thumbnail URL, and any short summary text.

To detect when to stop, I track whether any new titles appear on each page. If three consecutive pages return no new results the scraper stops. Results are written to CSV after each page via a temp file swap so progress is not lost if something fails mid-run.

In [ ]:
def safe_write_csv(rows, path):
    tmp = path + ".tmp"
    pd.DataFrame(rows).to_csv(tmp, index=False)
    os.replace(tmp, path)


def scrape_british_pathe_turin():
    driver         = make_driver(HEADLESS)
    all_data       = []
    seen_titles    = set()
    page           = 1
    repeated_pages = 0
    max_repeats    = 3

    try:
        while True:
            url = f"{BASE_URL}&page={page}"
            print(f"Loading page {page}: {url}")
            driver.get(url)

            try:
                WebDriverWait(driver, 25).until(
                    EC.presence_of_all_elements_located(
                        (By.CSS_SELECTOR, "div.card-v3")
                    )
                )
            except TimeoutException:
                print("No content detected on this page — stopping.")
                break

            cards = driver.find_elements(By.CSS_SELECTOR, "div.card-v3")
            if not cards:
                print("No more cards found.")
                break

            before = len(all_data)
            print(f"  Found {len(cards)} cards on page {page}")

            for card in cards:
                try:
                    title = card.find_element(
                        By.CSS_SELECTOR, "input.assetCheckbox"
                    ).get_attribute("data-name") or ""

                    if title in seen_titles:
                        continue
                    seen_titles.add(title)

                    video_page_url, mp4_url, thumb, summary = "", "", "", ""

                    try:
                        video_page_url = card.find_element(
                            By.CSS_SELECTOR, "a.card-v3__link"
                        ).get_attribute("href") or ""
                    except Exception:
                        pass
                    try:
                        mp4_url = card.find_element(
                            By.CSS_SELECTOR, "video source"
                        ).get_attribute("src") or ""
                    except Exception:
                        pass
                    try:
                        thumb = card.find_element(
                            By.CSS_SELECTOR, "video"
                        ).get_attribute("poster") or ""
                    except Exception:
                        pass
                    try:
                        summary = card.find_element(
                            By.CSS_SELECTOR, ".card-v3__content, p.font-italic, p"
                        ).text.strip()
                    except Exception:
                        pass

                    all_data.append({
                        "title":          title,
                        "video_page_url": video_page_url,
                        "mp4_url":        mp4_url,
                        "thumbnail":      thumb,
                        "short_summary":  summary
                    })
                    print(f"  [{len(all_data)}] {title[:80]}")

                except Exception as e:
                    print(f"  Card error: {e}")
                    continue

            safe_write_csv(all_data, OUTPUT_CSV)
            print(f"  Saved {len(all_data)} rows")

            after = len(all_data)
            if after == before:
                repeated_pages += 1
                if repeated_pages >= max_repeats:
                    print("  Results repeating — scraper finished.")
                    break
            else:
                repeated_pages = 0

            page += 1
            time.sleep(1.5)

    finally:
        driver.quit()

    print(f"Total unique videos scraped: {len(all_data)}")
    return pd.DataFrame(all_data)


df = scrape_british_pathe_turin()
df.head()

## 4. Downloading thumbnails

Each row in the CSV has a thumbnail URL pointing to a still frame from the clip. This cell downloads them all to the `thumbnails_turin/` folder, naming each file by its row index. Files that already exist are skipped so the cell can be re-run safely.

In [ ]:
def download_thumbnails(csv_path=OUTPUT_CSV, out_dir=THUMBS_DIR):
    df   = pd.read_csv(csv_path)
    ok, fail = 0, 0

    for i, row in df.iterrows():
        url = str(row.get("thumbnail", "")).strip()
        if not url.startswith("http"):
            continue
        fname = os.path.join(out_dir, f"{i:03d}.jpg")
        if os.path.exists(fname):
            ok += 1
            continue
        try:
            r = requests.get(url, timeout=20)
            if r.status_code == 200:
                with open(fname, "wb") as f:
                    f.write(r.content)
                ok += 1
            else:
                fail += 1
        except Exception as e:
            print(f"  Error: {e}")
            fail += 1

    print(f"Thumbnails done. Success: {ok} | Failed: {fail}")


download_thumbnails()

## 5. Downloading videos

Many of the direct `.mp4` links return HTTP 403 if accessed without authentication. To work around this, I open each video page in Selenium, let the browser authenticate, then extract the actual video source URL from the `<video>` element. I copy the browser's session cookies and pass them to a standard `requests` download — this makes the request appear to come from an authenticated browser session. A 5-second wait ensures the video source element has fully rendered before reading it.

In [ ]:
def download_videos(csv_path=OUTPUT_CSV, out_dir=VIDEOS_DIR):
    df     = pd.read_csv(csv_path)
    driver = make_driver(HEADLESS)
    ok, fail = 0, 0

    try:
        for i, row in df.iterrows():
            page_url = str(row.get("video_page_url", "")).strip()
            title    = re.sub(r'[\\/:*?"<>|]', "_", str(row.get("title", f"video_{i}")).strip())
            fname    = os.path.join(out_dir, f"{i:03d}_{title[:50]}.mp4")

            if not page_url.startswith("http") or os.path.exists(fname):
                ok += 1
                continue

            print(f"[{i+1}/{len(df)}] {page_url}")
            driver.get(page_url)

            try:
                WebDriverWait(driver, 30).until(
                    EC.presence_of_element_located((By.TAG_NAME, "video"))
                )
                time.sleep(5)
                video_url   = driver.find_element(
                    By.CSS_SELECTOR, "video source"
                ).get_attribute("src")
                cookie_dict = {c["name"]: c["value"] for c in driver.get_cookies()}

                with requests.get(
                    video_url,
                    headers={"User-Agent": "Mozilla/5.0"},
                    cookies=cookie_dict,
                    stream=True, timeout=90
                ) as r:
                    if r.status_code == 200:
                        with open(fname, "wb") as f:
                            for chunk in r.iter_content(chunk_size=8192):
                                if chunk:
                                    f.write(chunk)
                        ok += 1
                        print(f"  Saved: {fname}")
                    else:
                        print(f"  HTTP {r.status_code}")
                        fail += 1
            except Exception as e:
                print(f"  Error: {e}")
                fail += 1

            time.sleep(2)

    finally:
        driver.quit()

    print(f"Videos done. Success: {ok} | Failed: {fail}")


download_videos()

## 6. Dataset summary

In [ ]:
df = pd.read_csv(OUTPUT_CSV)
print(f"Total clips  : {len(df)}")
print(f"Columns      : {list(df.columns)}")
print(f"Thumbnails   : {df['thumbnail'].notna().sum()} URLs collected")
print(f"MP4 links    : {df['mp4_url'].notna().sum()} URLs collected")
print(f"With summary : {df['short_summary'].notna().sum()} clips")
print()
df.head(10)